[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pymatchmaker/mec2026_alignment_workshop/blob/main/offline_alignment/MEC2026_audio_audio_alignment.ipynb)

# MEC 2026 — Audio-to-Audio Offline Alignment

**Music Alignment Workshop, MEC 2026 — Section 2 (Offline Alignment)**

In this notebook we align **two recordings of the same piece** using the [Sync Toolbox](https://github.com/meinardmueller/synctoolbox). The pipeline is:

1. Load two performances of the same piece.
2. Estimate tuning and compute **chroma features** (and *hear* them via [libsoni](https://github.com/groupmm/libsoni)).
3. Detect a possible **transposition** (key difference) between the two recordings.
4. Align them with **MrMsDTW** (multi-resolution multi-scale dynamic time warping).
5. **Sonify** the result: warp version 1 to run synchronously with version 2 (time-scale modification (TSM) via [libtsm](https://github.com/groupmm/libtsm)).
6. **Export** the alignment (warping-path CSV + Sonic Visualiser layers).

Our example piece is the first variation in Mozart's *"Ah! vous dirai-je, maman"* (K. 265), with two performances at **different tempi** (a slower and a faster reading) — a clear illustration of what time-warping has to undo. You can also swap in your **own** two recordings (upload files or paste YouTube links) — see the data-loading section.

> **Note.** To keep things focused, this notebook uses **chroma features only**. The Sync Toolbox can additionally use onset-based *DLNCO* features for higher accuracy on music with clear onsets — see [`sync_audio_audio_full.ipynb`](https://github.com/meinardmueller/synctoolbox/blob/master/sync_audio_audio_full.ipynb).

## 0. Setup

Install the Sync Toolbox, `libtsm` (sonification by time-scale modification), `libsoni` (feature sonification) and `yt-dlp` (optional YouTube download), then import everything we need.

In [ ]:
# synctoolbox's feature.visualization module isn't in the PyPI 1.4.2 release yet, so we install
# synctoolbox from GitHub. Both are version 1.4.2, so pip would otherwise treat the PyPI copy as
# "already satisfied" and skip the GitHub install -- hence the explicit --force-reinstall.
!pip install -q synctoolbox librosa libtsm libsoni yt-dlp
!pip install -q --force-reinstall --no-deps "synctoolbox @ git+https://github.com/groupmm/synctoolbox.git"

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import IPython.display as ipd

from synctoolbox.feature.visualization import plot_chromagram, plot_signal
from synctoolbox.dtw.mrmsdtw import sync_via_mrmsdtw
from synctoolbox.dtw.utils import compute_optimal_chroma_shift, shift_chroma_vectors, make_path_strictly_monotonic
from synctoolbox.feature.chroma import pitch_to_chroma, quantize_chroma, quantized_chroma_to_CENS
from synctoolbox.feature.pitch import audio_to_pitch_features
from synctoolbox.feature.utils import estimate_tuning
%matplotlib inline

# Global parameters
Fs = 22050              # sampling rate
feature_rate = 50       # feature frames per second
step_weights = np.array([1.5, 1.5, 2.0])
threshold_rec = 10 ** 6
figsize = (9, 3)

## 1. Load two recordings of the same piece

We need **two performances of the same piece**. By default we use the provided Mozart example.

**To use your own recordings:** either upload two audio files when prompted, set `DATA_BASE_URL` / the two filenames below, or use the YouTube option further down. Any common audio format works (`.wav`, `.mp3`, `.flac`, ...).

In [ ]:
import os

# --- Configure your two recordings here -------------------------------------
# DATA_BASE_URL points at this repo so Colab can auto-download the bundled (open) example
# audio. Run locally and the files in data_music/ are found directly (no download).
DATA_BASE_URL = "https://raw.githubusercontent.com/pymatchmaker/mec2026_alignment_workshop/main/offline_alignment/"
audio_1_filename = "data_music/Mozart_KV265_short_slow1.wav"   # version 1 (slower performance)
audio_2_filename = "data_music/Mozart_KV265_short_fast1.wav"   # version 2 (faster performance)
# ---------------------------------------------------------------------------


def fetch_or_upload(filename):
    """Return a local path to `filename`: use it if present, else download from DATA_BASE_URL,
    else fall back to a manual upload (Colab)."""
    if os.path.exists(filename):
        return filename
    if DATA_BASE_URL:
        import urllib.request
        url = DATA_BASE_URL.rstrip('/') + '/' + filename
        try:
            print(f'Downloading {url} ...')
            os.makedirs(os.path.dirname(filename) or '.', exist_ok=True)
            urllib.request.urlretrieve(url, filename)
            return filename
        except Exception as e:
            print(f'  Could not download ({e}); falling back to manual upload.')
    try:
        from google.colab import files
        print(f'Please upload: {os.path.basename(filename)}')
        uploaded = files.upload()
        return list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f'{filename!r} not found. Set DATA_BASE_URL or place the file next to the notebook.')


audio_1_path = fetch_or_upload(audio_1_filename)
audio_2_path = fetch_or_upload(audio_2_filename)

### (Alternative) Load from YouTube

Prefer your **own** recordings? Paste two YouTube links below (two performances of the *same* piece) and run these two cells **instead of** the upload cell above — they set `audio_1_path` / `audio_2_path` the same way, so the rest of the notebook is unchanged.

> Tip: trimming to a common excerpt (e.g. the first 30 s) keeps the demo fast. Please use short excerpts and respect copyright.

In [ ]:
import subprocess


def download_youtube_audio(url, out_basename, start_sec=None, end_sec=None):
    """Download audio from a YouTube URL to a WAV file (only the [start_sec, end_sec] section
    if both are given). Returns the path to the WAV.

    Raises RuntimeError with yt-dlp's output on failure — most often YouTube's "confirm you're
    not a bot" check, which blocks Colab's datacenter IPs. If that happens, pre-download the
    excerpt and upload it (or place it next to the notebook) instead.
    """
    cmd = ['yt-dlp', '-x', '--audio-format', 'wav', '--force-overwrites']
    if start_sec is not None and end_sec is not None:
        cmd += ['--download-sections', f'*{start_sec}-{end_sec}']
    cmd += ['-o', out_basename + '.%(ext)s', url]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError((result.stderr or result.stdout or 'yt-dlp failed').strip())
    return out_basename + '.wav'

In [ ]:
# Two performances of the SAME piece, as YouTube links.
youtube_url_1 = 'https://www.youtube.com/watch?v=XXXXXXXXXXX'
youtube_url_2 = 'https://www.youtube.com/watch?v=YYYYYYYYYYY'

# Optional trim in seconds as (start, end); use None for the full track.
trim_1 = None   # e.g. (0, 30)
trim_2 = None   # e.g. (0, 30)

audio_1_path = download_youtube_audio(youtube_url_1, 'youtube_version1', *(trim_1 or (None, None)))
audio_2_path = download_youtube_audio(youtube_url_2, 'youtube_version2', *(trim_2 or (None, None)))
print('Saved:', audio_1_path, 'and', audio_2_path)

### Version 1

In [ ]:
audio_1, _ = librosa.load(audio_1_path, sr=Fs)

plot_signal(audio_1, Fs=Fs, ylabel='Amplitude', title='Version 1', figsize=figsize)
ipd.display(ipd.Audio(audio_1, rate=Fs))

### Version 2

In [ ]:
audio_2, _ = librosa.load(audio_2_path, sr=Fs)

plot_signal(audio_2, Fs=Fs, ylabel='Amplitude', title='Version 2', figsize=figsize)
ipd.display(ipd.Audio(audio_2, rate=Fs))

## 2. Estimate tuning

Recordings are often tuned slightly differently (e.g. not exactly A=440 Hz). We estimate each recording's tuning deviation and feed it into the feature extraction — otherwise the chroma features can look "smeared" and degrade the alignment.

In [ ]:
tuning_offset_1 = estimate_tuning(audio_1, Fs)
tuning_offset_2 = estimate_tuning(audio_2, Fs)
print('Estimated tuning deviation — version 1: %d cents, version 2: %d cents' % (tuning_offset_1, tuning_offset_2))

## 3. Compute chroma features

We compute **quantized chroma** features for each recording. Chroma captures the harmonic/melodic content (the 12 pitch classes) while being robust to timbre and instrumentation — ideal for comparing two different performances.

Internally, MrMsDTW further smooths, downsamples and normalizes these (CENS-style), so we just hand it the quantized chroma.

In [ ]:
def get_chroma_from_audio(audio, tuning_offset, visualize=False):
    f_pitch = audio_to_pitch_features(f_audio=audio, Fs=Fs, tuning_offset=tuning_offset,
                                      feature_rate=feature_rate, verbose=visualize)
    f_chroma = pitch_to_chroma(f_pitch=f_pitch)
    f_chroma_quantized = quantize_chroma(f_chroma=f_chroma)
    return f_chroma_quantized


f_chroma_quantized_1 = get_chroma_from_audio(audio_1, tuning_offset_1)
f_chroma_quantized_2 = get_chroma_from_audio(audio_2, tuning_offset_2)

In [ ]:
plot_chromagram(f_chroma_quantized_1[:, :30 * feature_rate], Fs=feature_rate,
                title='Chroma — version 1 (first 30 s)', figsize=figsize)
plt.show()
plot_chromagram(f_chroma_quantized_2[:, :30 * feature_rate], Fs=feature_rate,
                title='Chroma — version 2 (first 30 s)', figsize=figsize)
plt.show()

### Hear the chroma features

Chroma can also be **listened to**, not just plotted. Using [libsoni](https://github.com/groupmm/libsoni), we render each chroma sequence as a stack of *Shepard tones* — so you hear the harmonic content (the 12 pitch classes over time) that the alignment is based on, stripped of timbre and octave register.

In [ ]:
import libsoni

# libsoni expects chroma as 12 x N (synctoolbox's native layout) and a hop size H.
# Our feature rate of 50 fps corresponds to a hop of Fs / feature_rate samples.
H = Fs // feature_rate 

chroma_sonification_1 = libsoni.sonify_chromagram(f_chroma_quantized_1, H=H, fs=Fs)

print('Chroma sonification — version 1', flush=True)
ipd.display(ipd.Audio(chroma_sonification_1, rate=Fs, normalize=True))

## 4. Detect transposition (key difference)

Two performances may be played in **different keys**. If so, their chroma sequences are circularly shifted relative to each other, which would wreck the alignment. The Sync Toolbox finds the optimal chroma shift (by trying all 12 shifts and picking the lowest-cost one) on a coarse, downsampled version of the features.

For our Mozart example the shift is typically 0 (same key), but this step makes the notebook robust to any pair of recordings you throw at it.

In [ ]:
f_cens_1hz_1 = quantized_chroma_to_CENS(f_chroma_quantized_1, 201, 50, feature_rate)[0]
f_cens_1hz_2 = quantized_chroma_to_CENS(f_chroma_quantized_2, 201, 50, feature_rate)[0]
opt_chroma_shift = compute_optimal_chroma_shift(f_cens_1hz_1, f_cens_1hz_2)
print('Optimal chroma shift (version 2 relative to version 1):', opt_chroma_shift, 'semitone bins')

f_chroma_quantized_2 = shift_chroma_vectors(f_chroma_quantized_2, opt_chroma_shift)

## 5. Align with MrMsDTW

Now we align the two chroma sequences. **MrMsDTW** aligns on progressively finer resolutions, each level constrained by the alignment found on the coarser level above it — this is what makes it fast and memory-efficient enough for full pieces.

The result is a **warping path** `wp`: a sequence of index pairs `[n, m]` saying "frame *n* of version 1 corresponds to frame *m* of version 2".

In [ ]:
wp = sync_via_mrmsdtw(f_chroma1=f_chroma_quantized_1,
                      f_chroma2=f_chroma_quantized_2,
                      input_feature_rate=feature_rate,
                      step_weights=step_weights,
                      threshold_rec=threshold_rec,
                      verbose=True)

### Make the warping path strictly monotonic

The standard DTW step sizes allow horizontal/vertical steps, so the raw path can briefly "stand still" in one recording. For applications like sonification (and annotation transfer) it is cleaner to enforce a **strictly monotonic** path, interpolating linearly through any non-monotonic segments.

In [ ]:
print('Warping path length (raw):', wp.shape[1])
wp = make_path_strictly_monotonic(wp)
print('Warping path length (strictly monotonic):', wp.shape[1])

## 6. Sonify the alignment (TSM)

To *hear* the alignment, we time-scale version 1 (using the warping path) so that it runs **synchronously** with version 2. If the two recordings were in different keys, we also pitch-shift version 1 to match. We then put the warped version 1 in the **left** channel and version 2 in the **right** channel of a stereo file. If the alignment is good, the two performances stay locked together throughout.

We use [`libtsm`](https://github.com/meinardmueller/libtsm) for the time-scale modification.

In [ ]:
import libtsm

# Pitch-shift version 1 to match version 2's key (no-op if same key).
pitch_shift_for_audio_1 = -opt_chroma_shift % 12
if pitch_shift_for_audio_1 > 6:
    pitch_shift_for_audio_1 -= 12
audio_1_shifted = libtsm.pitch_shift(audio_1, pitch_shift_for_audio_1 * 100, order='tsm-res')

# libtsm expects the warping path in audio samples; convert and clip.
time_map = wp.T / feature_rate * Fs
time_map[time_map[:, 0] > len(audio_1), 0] = len(audio_1) - 1
time_map[time_map[:, 1] > len(audio_2), 1] = len(audio_2) - 1

y_hpstsm = libtsm.hps_tsm(audio_1_shifted, time_map)

# The warped signal can differ from version 2 by a sample or two; trim both to the common length.
L = min(len(audio_2), y_hpstsm.shape[0])
stereo_sonification = np.column_stack((audio_2[:L], y_hpstsm[:L].reshape(-1)))

In [ ]:
print('Original version 1', flush=True)
ipd.display(ipd.Audio(audio_1, rate=Fs, normalize=True))

print('Original version 2', flush=True)
ipd.display(ipd.Audio(audio_2, rate=Fs, normalize=True))

print('Synchronized: version 2 — reference (left) + version 1 — time-warped (right)', flush=True)
ipd.display(ipd.Audio(stereo_sonification.T, rate=Fs, normalize=True))

## 7. Export the alignment (CSV + Sonic Visualiser)

Finally, we save the alignment for use outside this notebook.

- **`warping_path.csv`** — the full warping path as corresponding times (seconds) in the two recordings.
- **`sonic_visualiser_version1.csv` / `_version2.csv`** — sparse correspondence layers for [Sonic Visualiser](https://www.sonicvisualiser.org/): open one recording's audio, then *File ▸ Import Annotation Layer…* and pick the matching CSV. The first column is the time (seconds) in that recording; each point is labelled with the corresponding time in the other recording.

In [ ]:
# Full warping path, in seconds
wp_seconds = wp / feature_rate
pd.DataFrame({'time_version1_sec': wp_seconds[0],
              'time_version2_sec': wp_seconds[1]}).to_csv('warping_path.csv', index=False)
print('Wrote warping_path.csv with', wp.shape[1], 'points')

In [ ]:
# Sparse correspondence layers for Sonic Visualiser (one point every `sv_interval_sec`).
sv_interval_sec = 1.0
t1, t2 = wp_seconds[0], wp_seconds[1]

# Load this over VERSION 2: time = position in v2, label = matching position in v1
grid_v2 = np.arange(t2[0], t2[-1], sv_interval_sec)
pd.DataFrame({'time': grid_v2, 'label': np.round(np.interp(grid_v2, t2, t1), 3)}).to_csv(
    'sonic_visualiser_version2.csv', index=False, header=False)

# Load this over VERSION 1: time = position in v1, label = matching position in v2
grid_v1 = np.arange(t1[0], t1[-1], sv_interval_sec)
pd.DataFrame({'time': grid_v1, 'label': np.round(np.interp(grid_v1, t1, t2), 3)}).to_csv(
    'sonic_visualiser_version1.csv', index=False, header=False)

print('Wrote sonic_visualiser_version1.csv and sonic_visualiser_version2.csv')

In [ ]:
# In Colab, uncomment to download the CSV files to your computer:
# from google.colab import files
# for f in ['warping_path.csv', 'sonic_visualiser_version1.csv', 'sonic_visualiser_version2.csv']:
#     files.download(f)

## 8. Bonus: aligning a piano reduction with the orchestra

A more striking case is **cross-instrument** alignment: matching an orchestral recording with a **piano reduction** of the same work. Here we use **Tchaikovsky's *The Nutcracker* Suite, Op. 71a** — an orchestral recording and a piano-reduction recording of the same movement, both in the public domain and bundled with this notebook.

The pipeline is *exactly* the same (chroma → MrMsDTW → TSM); only the inputs differ. Despite the huge timbral gap between a piano and a full orchestra, chroma captures the shared harmonic content well enough to align them.

We sonify the alignment **both ways**, which highlights what *reference* really means: **TSM only alters the signal being warped — the reference keeps its original audio.** With the piano as reference the orchestra is bent (and picks up the time-stretch artifacts); with the orchestra as reference (the original work) the piano is bent instead. Listen for where the artifacts land.

The cell below packs the whole pipeline into one helper, `synchronize_and_sonify`, and `show_both_directions` runs it each way.

In [ ]:
import libtsm


def synchronize_and_sonify(warp_audio, reference_audio):
    """Align two recordings and warp `warp_audio` onto the timeline of `reference_audio`.

    Returns a stereo sonification (reference left, time-warped `warp_audio` right) and the
    warping path. This packs the exact pipeline from the sections above into one call:
    tuning -> chroma -> transposition shift -> MrMsDTW -> strictly monotonic path -> TSM.
    """
    c_w = get_chroma_from_audio(warp_audio, estimate_tuning(warp_audio, Fs))
    c_r = get_chroma_from_audio(reference_audio, estimate_tuning(reference_audio, Fs))

    # Account for a possible key difference (typically 0 for the same work).
    cens_w = quantized_chroma_to_CENS(c_w, 201, 50, feature_rate)[0]
    cens_r = quantized_chroma_to_CENS(c_r, 201, 50, feature_rate)[0]
    shift = compute_optimal_chroma_shift(cens_w, cens_r)
    c_r = shift_chroma_vectors(c_r, shift)

    wp = sync_via_mrmsdtw(f_chroma1=c_w, f_chroma2=c_r, input_feature_rate=feature_rate,
                          step_weights=step_weights, threshold_rec=threshold_rec, verbose=False)
    wp = make_path_strictly_monotonic(wp)

    pitch_shift = -shift % 12
    if pitch_shift > 6:
        pitch_shift -= 12
    warp_shifted = libtsm.pitch_shift(warp_audio, pitch_shift * 100, order='tsm-res')

    time_map = wp.T / feature_rate * Fs
    time_map[time_map[:, 0] > len(warp_audio), 0] = len(warp_audio) - 1
    time_map[time_map[:, 1] > len(reference_audio), 1] = len(reference_audio) - 1
    y_warped = libtsm.hps_tsm(warp_shifted, time_map)

    L = min(len(reference_audio), y_warped.shape[0])
    stereo = np.column_stack((reference_audio[:L], y_warped[:L].reshape(-1)))
    return stereo, wp


def load_audio(local_path=None, youtube_url=None, start=None, end=None, cache_name=None):
    """Load one recording, in order of preference: a local file, else a (cached) YouTube
    download of the [start, end] excerpt, else an upload/hosted download via `fetch_or_upload`.

    If a YouTube download fails (e.g. YouTube's bot check on Colab), we fall back to
    `fetch_or_upload` so you can upload an excerpt instead of the notebook crashing.
    """
    if local_path and os.path.exists(local_path):
        return librosa.load(local_path, sr=Fs)[0]
    if youtube_url is not None:
        out = cache_name or 'recording'
        path = out + '.wav'
        if not os.path.exists(path):
            try:
                download_youtube_audio(youtube_url, out, start, end)
            except Exception as e:
                print(f'  YouTube download failed: {e}')
                print('  Falling back to a local file / upload ...')
                return librosa.load(fetch_or_upload(local_path), sr=Fs)[0]
        return librosa.load(path, sr=Fs)[0]
    return librosa.load(fetch_or_upload(local_path), sr=Fs)[0]


def show_both_directions(piano, orchestra, work_name):
    """Sonify a piano/orchestra alignment with each side used as the reference timeline.

    Remember: TSM only alters the warped signal; the reference keeps its original audio.
    """
    print(f'=== {work_name} ===', flush=True)
    print('Piano (original)', flush=True)
    ipd.display(ipd.Audio(piano, rate=Fs, normalize=True))
    print('Orchestra (original)', flush=True)
    ipd.display(ipd.Audio(orchestra, rate=Fs, normalize=True))

    stereo_piano_ref, _ = synchronize_and_sonify(warp_audio=orchestra, reference_audio=piano)
    print('Piano as reference: piano (left) + TSM-warped orchestra (right)', flush=True)
    ipd.display(ipd.Audio(stereo_piano_ref.T, rate=Fs, normalize=True))

    stereo_orch_ref, _ = synchronize_and_sonify(warp_audio=piano, reference_audio=orchestra)
    print('Orchestra as reference: orchestra (left) + TSM-warped piano (right)', flush=True)
    ipd.display(ipd.Audio(stereo_orch_ref.T, rate=Fs, normalize=True))

### Example: Tchaikovsky, *The Nutcracker* Suite, Op. 71a (piano reduction ↔ orchestra)

In [ ]:
# Tchaikovsky, The Nutcracker Suite, Op. 71a:
# an orchestral recording vs. a piano-reduction recording of the same movement.
# Both excerpts are public-domain and bundled in data_music/ (auto-downloaded on Colab).
piano_tchaikovsky = load_audio(
    local_path='data_music/Tchaikovsky_Op071a-02-a_HernandezRomero2016_IMSLP_P_short.wav',
    cache_name='Tchaikovsky_Op071a_piano')
orch_tchaikovsky = load_audio(
    local_path='data_music/Tchaikovsky_Op071a-02-a_Bernstein1960_IMSLP_O_short.wav',
    cache_name='Tchaikovsky_Op071a_orch')

show_both_directions(piano_tchaikovsky, orch_tchaikovsky, 'Tchaikovsky, The Nutcracker Suite, Op. 71a')

## 9. Try it yourself

A few things to experiment with — re-run from the alignment step after each change:

- **Your own recordings.** Go back to the data-loading section and supply two performances of *your* favourite piece. Upload files, or paste two **YouTube** links. Do they stay locked together in the sonification?
- **Piano vs. orchestra.** Reuse `synchronize_and_sonify` from Section 8 on another arrangement pair. Does chroma cope with the timbre gap?
- **Different keys.** Try two recordings known to be in different keys and watch the *optimal chroma shift* in Step 4.
- **Step weights.** Change `step_weights` (e.g. `np.array([1.0, 1.0, 1.0])`). Diagonal vs. horizontal/vertical preference changes how "willing" the path is to stretch time.

## References

[1] Meinard Müller, Yigitcan Özer, Michael Krause, Thomas Prätzlich, and Jonathan Driedger: Sync Toolbox: A Python Package for Efficient, Robust, and Accurate Music Synchronization, JOSS, 2021.

[2] Thomas Prätzlich, Jonathan Driedger, and Meinard Müller: Memory-Restricted Multiscale Dynamic Time Warping, ICASSP, 2016.

[3] Meinard Müller, Henning Mattes, and Frank Kurth: An Efficient Multiscale Approach to Audio Synchronization, ISMIR, 2006.

[4] Meinard Müller: Information Retrieval for Music and Motion, Springer, 2007.

[5] Sebastian Rosenzweig, Simon Schwär, Jonathan Driedger, and Meinard Müller: Adaptive Pitch-Shifting with Applications to Intonation Adjustment in A Cappella Recordings, DAFx, 2021.

[6] Yigitcan Özer, Leo Brütting, Simon Schwär, and Meinard Müller: libsoni: A Python Toolbox for Sonifying Music Annotations and Feature Representations, JOSS, 2024.